## Hacking 4 - Deepseek MLA Walkthrough simplified
- Ignore setup and naive method right now, got back to these as needed
- Here just focus on understanding what's happening with the MLA forward pass.
- Oh wow looks like it might kinda be possible to just load a subset of data from the safetensors?!
- Hmm that's pretty interesting - distraction or a good use of time? Let me poke at it for a couplte minutes here.
- If i wanted all the layer6 atention stuff, it looks like I would just need

```
"model.layers.6.self_attn.q_a_proj.weight": "model-00009-of-000163.safetensors"
```

And layer 0 attention stuff would just be: 

```
"model.layers.0.self_attn.q_a_proj.weight": "model-00001-of-000163.safetensors"
```
- Ok I have a theory about why I can't load weights -> I bet it's because the hf and github repos -> eh that doesn't seem to be it. 
- Tried manually mapping too -> ran into size issues.
- Ok let me push a little further here and try a "clean start" from the hf repo"
- Ok yeah I think the step i've been missing is "convert.py"

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from transformers import AutoTokenizer
import torch

import sys
# sys.path.append('DeepSeek-V3/inference') #Github slightly newer
sys.path.append('DeepSeek-V3/DeepSeek-V3/inference') #Hugging face

from model import Transformer, MLA, ModelArgs, apply_rotary_emb

In [ ]:
args=ModelArgs(**{"vocab_size": 129280,
                "dim": 7168,
                "inter_dim": 18432,
                "moe_inter_dim": 2048,
                "n_layers": 61,
                "n_dense_layers": 3,
                "n_heads": 128,
                "n_routed_experts": 256,
                "n_shared_experts": 1,
                "n_activated_experts": 8,
                "n_expert_groups": 8,
                "n_limited_groups": 4,
                "route_scale": 2.5,
                "score_func": "sigmoid",
                "q_lora_rank": 1536,
                "kv_lora_rank": 512,
                "qk_nope_head_dim": 128,
                "qk_rope_head_dim": 64,
                "v_head_dim": 128,
                "dtype": "bf16"}) #fp8 seems out due to my hardware? bf16

In [ ]:
model=Transformer(args)

### Let me spend a few minutes seeing if I can load just the attention weights for the embedding, layers 0 and maybe layer 6

In [ ]:
from safetensors.torch import load_model, load_file
import os

In [ ]:
# ckpt_path='../data/deepseek/DeepSeek-V3/DeepSeek-V3'
ckpt_path='../data/deepseek/clean_hf_pull/DeepSeek-V3/weights_converted'
# rank=0
# world_size=1

#
# missing, unexpected=load_model(model, os.path.join(ckpt_path, 'model-00001-of-000163.safetensors'), strict=False)
# load_model(model, os.path.join(ckpt_path, 'model-00009-of-000163.safetensors'), strict=False)
missing, unexpected=load_model(model, os.path.join(ckpt_path, "model0-mp1.safetensors"), strict=False) #Result of conversion process

In [ ]:
len(missing)

In [ ]:
len(unexpected)

In [ ]:
state_dict = load_file(os.path.join(ckpt_path, "model0-mp1.safetensors"), device='cpu')

In [ ]:
state_dict.keys()

In [ ]:
model_state_dict=model.state_dict()

In [ ]:
# model_state_dict.keys()

In [ ]:
len(state_dict)

In [ ]:
missing, unexpected = model.load_state_dict(state_dict, strict=False)

In [ ]:
len(unexpected) #Ok some are getting loaded -> maybe the ones I need are?

In [ ]:
model.layers[0].attn.wq_a.weight.detach().abs().sum()

Numbers!!!

In [ ]:
model.layers[0].attn.wq_a.weight

In [ ]:
# model.layers[0].attn.wq_a.bias

In [ ]:
model.embed.weight

- OK NICE!
- Took some hacking, but I should be able to trace like actually values through this thing now
- Next I'll convert a few more tensors (into a new directory), make a clean from scratch notebook, and get to tracing and visualizing! Let's go!

`load_model` not quite working as expected, getting a little deeper. 
```
    state_dict = load_file(filename, device=device)
    model_state_dict = model.state_dict()
    to_removes = _remove_duplicate_names(model_state_dict, preferred_names=state_dict.keys())
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
```

In [ ]:
state_dict = load_file(os.path.join(ckpt_path, 'model-00001-of-000163.safetensors'), device='cpu')

In [ ]:
model_state_dict=model.state_dict()

In [ ]:
#Statedict from safe tensor
'model.embed_tokens.weight', 
'model.layers.0.input_layernorm.weight', 
'model.layers.0.mlp.down_proj.weight', 
'model.layers.0.mlp.down_proj.weight_scale_inv', 
'model.layers.0.mlp.gate_proj.weight', 
'model.layers.0.mlp.gate_proj.weight_scale_inv', 
'model.layers.0.mlp.up_proj.weight',
'model.layers.0.mlp.up_proj.weight_scale_inv', 
'model.layers.0.post_attention_layernorm.weight', 

'model.layers.0.self_attn.kv_a_layernorm.weight', 
'model.layers.0.self_attn.kv_a_proj_with_mqa.weight', 
'model.layers.0.self_attn.kv_a_proj_with_mqa.weight_scale_inv', 
'model.layers.0.self_attn.kv_b_proj.weight', 
'model.layers.0.self_attn.kv_b_proj.weight_scale_inv', 

'model.layers.0.self_attn.o_proj.weight', 
'model.layers.0.self_attn.o_proj.weight_scale_inv',

'model.layers.0.self_attn.q_a_layernorm.weight', 
'model.layers.0.self_attn.q_a_proj.weight', 
'model.layers.0.self_attn.q_a_proj.weight_scale_inv', 
'model.layers.0.self_attn.q_b_proj.weight', 
'model.layers.0.self_attn.q_b_proj.weight_scale_inv',  #Hmm not sure where to put these

'model.layers.1.input_layernorm.weight'

In [ ]:
#Statedict from model -> these names are super different dafuq? Man I hope I'm not missing something super essential
'embed.weight', 
'layers.0.attn.wq_a.weight', 
'layers.0.attn.q_norm.weight', 
'layers.0.attn.wq_b.weight', 
'layers.0.attn.wkv_a.weight', 
'layers.0.attn.kv_norm.weight', 
'layers.0.attn.wkv_b.weight', 
'layers.0.attn.wo.weight', 
'layers.0.ffn.w1.weight', 
'layers.0.ffn.w2.weight', 
'layers.0.ffn.w3.weight', 
'layers.0.attn_norm.weight', 
'layers.0.ffn_norm.weight'

In [ ]:
# Ok this seems sketchy -> I'm kinda having to guess which thing is which -> maybe i try from scratch using
# hf repo? Does seem to be sortof working, let me push a little further and fall back to that if needed
my_state_dict={}
my_state_dict['embed.weight']=state_dict['model.embed_tokens.weight']
my_state_dict['layers.0.attn.wq_a.weight']=state_dict['model.layers.0.self_attn.q_a_proj.weight']
my_state_dict['layers.0.attn.q_norm.weight']=state_dict['model.layers.0.self_attn.q_a_layernorm.weight']
my_state_dict['layers.0.attn.wq_b.weight']=state_dict['model.layers.0.self_attn.q_b_proj.weight']

my_state_dict['layers.0.attn.wkv_a.weight'] =state_dict['model.layers.0.self_attn.kv_a_layernorm.weight']
my_state_dict['layers.0.attn.kv_norm.weight'] =state_dict['model.layers.0.self_attn.kv_a_layernorm.weight']
my_state_dict['layers.0.attn.wkv_b.weight'] =state_dict['model.layers.0.self_attn.kv_b_proj.weight']
my_state_dict['layers.0.attn.wo.weight'] =state_dict['model.layers.0.self_attn.o_proj.weight']

# my_state_dict['layers.0.ffn.w1.weight'] =state_dict[]
# my_state_dict['layers.0.ffn.w2.weight'] =state_dict[]
# my_state_dict['layers.0.ffn.w3.weight'] =state_dict[]
# my_state_dict['layers.0.attn_norm.weight'] =state_dict[]
# my_state_dict['layers.0.ffn_norm.weight']=state_dict[]

In [ ]:
# state_dict.keys()

In [ ]:
# model_state_dict.keys()

In [ ]:
# state_dict.keys()

In [ ]:
len(state_dict)

In [ ]:
# state_dict['model.embed_tokens.weight']

In [ ]:
# missing, unexpected = model.load_state_dict(state_dict, strict=False) #, assign=True)
missing, unexpected = model.load_state_dict(my_state_dict, strict=False) #, assign=True)

In [ ]:
len(missing)

In [ ]:
len(unexpected)

In [ ]:
model.embed

In [ ]:
# model.load_state_dict?

In [ ]:
len(missing)

In [ ]:
len(unexpected)

In [ ]:
model.layers[0].attn.wq_a.weight.detach().abs().sum()

In [ ]:
# model.layers[6].attn.kv_a_layernorm.weight
# plt.imshow(model.embed.weight.detach().cpu().numpy())

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('../data/deepseek/DeepSeek-V3/DeepSeek-V3')

In [ ]:
tokens=tokenizer.encode("The American flag is red, white, and")
model.forward(torch.tensor([tokens]), start_pos=0) #Dope! Ran on CPU somehow - took a couple minutes. 

In [ ]:
#Do I have non-zero weights now?
model.layers[0].attn.wq_a.weight.shape

In [ ]:
model.layers[6].attn.wq_a.weight

In [ ]:
start_pos=0 #might need to noodle with this to understad caching
seqlen = torch.tensor([tokens]).size(1)
freqs_cis = model.freqs_cis[start_pos:start_pos+seqlen]

mask = None
if seqlen > 1:
    mask = torch.full((seqlen, seqlen), float("-inf"), device='cpu').triu_(1) #CPUing for now

start_pos, seqlen, freqs_cis.shape, mask.shape

In [ ]:
h = model.embed(torch.tensor([tokens]))

In [ ]:
h_out=model.layers[0].attn.forward(h, start_pos, freqs_cis, mask) #Run MLA forward pass from outside

In [ ]:
h_out.shape

### Now Set Through Forward Pass

In [ ]:
x=h
mla=model.layers[0].attn

In [ ]:
bsz, seqlen, _ = x.size()
end_pos = start_pos + seqlen
if mla.q_lora_rank == 0:
    q = self.wq(x)
else:
    q = mla.wq_b(mla.q_norm(mla.wq_a(x)))
    print('hay')

In [ ]:
#Query compression - probably not in script
h1=mla.wq_a(x)
h2=mla.q_norm(mla.wq_a(x))
q=mla.wq_b(mla.q_norm(mla.wq_a(x)))

In [ ]:
x.shape, h1.shape, h2.shape, q.shape

In [ ]:
q = q.view(bsz, seqlen, mla.n_local_heads, mla.qk_head_dim) #Ok splitting out the queries across all the heads
q_nope, q_pe = torch.split(q, [mla.qk_nope_head_dim, mla.qk_rope_head_dim], dim=-1) #Splitting queries into positional embeddings 
q_pe = apply_rotary_emb(q_pe, freqs_cis)

In [ ]:
q.shape, q_nope.shape, q_pe.shape

Ok, this is not that different from GPT-2, for each of our 128 heads, we haev a 10x128 query vector - got it. 

In [ ]:
kv = mla.wkv_a(x)

In [ ]:
x.shape, kv.shape

- Ok how does this compare to GPT-2 style?